# Coastal flood step 13: sector min/max map pairs with panel-specific scales

Creates comparison maps by sector with two panels per figure:
- left: minimum scenario
- right: maximum scenario

Each panel uses its **own robust scale** (q=0.995 of absolute avoided EAD) so patterns are easier to see.
Uses cached map layers from step 11 for speed.


In [ ]:
from pathlib import Path
import numpy
import pandas
import geopandas
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1 import make_axes_locatable

pandas.set_option('display.max_columns', 200)
pandas.set_option('display.width', 200)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)


In [ ]:
# Paths
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
cache_dir = base_path / 'dphil_paper_3/results_coastal_scenario_comparison/maps_min_max_comparison/cache'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

cache_files = {
    'minimum': cache_dir / 'minimum_all_sector_map_layer.geoparquet',
    'maximum': cache_dir / 'maximum_all_sector_map_layer.geoparquet',
}

for scenario_name, f in cache_files.items():
    if not f.exists():
        raise FileNotFoundError(
            f"Missing cache file for {scenario_name}: {f}\n"
            "Run coastal_flood_11 notebook once to create caches."
        )

if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')

print('Using cache files:')
for k, f in cache_files.items():
    print('-', k, '->', f)


In [ ]:
# Load cached layers
scenario_gdfs = {}
for scenario_name, f in cache_files.items():
    gdf = geopandas.read_parquet(f)
    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:3448')
    gdf['Avoided_EAD_USD'] = pandas.to_numeric(gdf['Avoided_EAD_USD'], errors='coerce').fillna(0.0)
    scenario_gdfs[scenario_name] = gdf
    print(f"{scenario_name}: {len(gdf):,} features")

jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs('EPSG:3448')


In [ ]:
# Config
display_quantile = 0.995
value_col = 'Avoided_EAD_USD'

sector_order = ['buildings', 'energy', 'transport', 'water']
available_sectors = sorted(set(scenario_gdfs['minimum']['Sector']).union(set(scenario_gdfs['maximum']['Sector'])))
sector_order = [s for s in sector_order if s in available_sectors] + [s for s in available_sectors if s not in sector_order]

# Optional manual cap override per scenario panel (USD): manual_caps_usd[sector][scenario] = cap
manual_caps_usd = {}
# Optional manual cap override per sector-shared scale (USD): manual_sector_cap_usd[sector] = cap
manual_sector_cap_usd = {}

# Sign-aware highlight colors
pos_outlier_color = '#145a32'  # high avoided
neg_outlier_color = '#8b1a1a'  # increased damage

# Base map colors
cmap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)

print('Sectors:', sector_order)


In [ ]:
def choose_units(cap_usd: float):
    if cap_usd >= 1e6:
        return 1e6, 'USD millions'
    if cap_usd >= 1e3:
        return 1e3, 'USD thousands'
    return 1.0, 'USD'


def compute_scenario_cap_usd(sector_gdf, sector_name, scenario_name):
    vals = sector_gdf[value_col].fillna(0.0)
    abs_vals = vals.abs()

    if len(abs_vals) == 0:
        cap_usd = 1.0
    else:
        cap_usd = float(abs_vals.quantile(display_quantile))
        if cap_usd <= 0:
            cap_usd = float(abs_vals.max()) if float(abs_vals.max()) > 0 else 1.0

    # Optional panel-level override
    cap_usd = float(manual_caps_usd.get(sector_name, {}).get(scenario_name, cap_usd))
    return cap_usd


def plot_sector_panel(ax, sector_gdf, sector_name, scenario_name, shared_cap_usd, unit_factor, unit_label):
    vals = sector_gdf[value_col].fillna(0.0)
    abs_vals = vals.abs()

    cap = shared_cap_usd / unit_factor

    sector_gdf = sector_gdf.copy()
    sector_gdf['_plot_val'] = (vals / unit_factor).clip(-cap, cap)
    sector_gdf['_is_outlier'] = abs_vals > shared_cap_usd

    norm = TwoSlopeNorm(vmin=-cap, vcenter=0.0, vmax=cap)

    # Base boundary
    jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

    geom_type = sector_gdf.geometry.geom_type.astype(str)
    polys = sector_gdf[geom_type.str.contains('Polygon', na=False)]
    lines = sector_gdf[geom_type.str.contains('LineString', na=False)]
    points = sector_gdf[geom_type.str.contains('Point', na=False)]

    if not polys.empty:
        polys.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.10, edgecolor='none', alpha=0.92, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.95, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, markersize=16, alpha=0.95, zorder=4)

    # Outlier overlays by sign
    outliers = sector_gdf[sector_gdf['_is_outlier']].copy()
    pos_outliers = outliers[outliers[value_col] > 0].copy()
    neg_outliers = outliers[outliers[value_col] < 0].copy()

    for subset, color in [(pos_outliers, pos_outlier_color), (neg_outliers, neg_outlier_color)]:
        if subset.empty:
            continue
        sgt = subset.geometry.geom_type.astype(str)
        sp = subset[sgt.str.contains('Polygon', na=False)]
        sl = subset[sgt.str.contains('LineString', na=False)]
        spt = subset[sgt.str.contains('Point', na=False)]

        if not sp.empty:
            sp.boundary.plot(ax=ax, color=color, linewidth=1.15, alpha=0.96, zorder=5)
        if not sl.empty:
            sl.plot(ax=ax, color=color, linewidth=2.0, alpha=0.96, zorder=5)
        if not spt.empty:
            spt.plot(ax=ax, color=color, markersize=36, alpha=0.96, zorder=5)

    # Label top outliers per sign
    label_n = 5
    top_pos = pos_outliers.assign(_abs=pos_outliers[value_col].abs()).sort_values('_abs', ascending=False).head(label_n)
    top_neg = neg_outliers.assign(_abs=neg_outliers[value_col].abs()).sort_values('_abs', ascending=False).head(label_n)
    labels = pandas.concat([top_pos, top_neg], ignore_index=True)

    if not labels.empty:
        rps = labels.geometry.representative_point()
        for (_, row), pt in zip(labels.iterrows(), rps):
            val_scaled = row[value_col] / unit_factor
            color = pos_outlier_color if row[value_col] > 0 else neg_outlier_color
            ax.text(
                pt.x,
                pt.y,
                f"{val_scaled:+,.1f}",
                fontsize=7,
                color=color,
                ha='left',
                va='bottom',
                zorder=6,
                bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': color, 'pad': 0.30}
            )

    # Tight extent
    minx, miny, maxx, maxy = jamaica_boundary.total_bounds
    pad_x = (maxx - minx) * 0.002
    pad_y = (maxy - miny) * 0.0005
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)

    # Panel title
    n_out = int(sector_gdf['_is_outlier'].sum())
    ax.set_title(f"{scenario_name.capitalize()} | shared sector scale +/-{cap:,.2f} {unit_label} | outliers: {n_out:,}", fontsize=11)
    ax.set_axis_off()

    # Panel colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('bottom', size='3.2%', pad=0.04)
    cbar = plt.colorbar(sm, cax=cax, orientation='horizontal')
    cbar.set_label(f'Avoided EAD ({unit_label})')

    return {
        'Scenario': scenario_name,
        'Sector': sector_name,
        'Scale_Cap_USD': shared_cap_usd,
        'Outlier_Features': n_out,
        'Features': int(len(sector_gdf)),
        'Outlier_Percent': 100.0 * n_out / len(sector_gdf) if len(sector_gdf) else numpy.nan,
    }, outliers


In [ ]:
# Generate per-sector min/max pair maps
out_dir = base_path / 'dphil_paper_3/results_coastal_scenario_comparison/maps_sector_min_max_pairs_panel_scales'
out_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []

for sector_name in sector_order:
    min_sector = scenario_gdfs['minimum'][scenario_gdfs['minimum']['Sector'] == sector_name].copy()
    max_sector = scenario_gdfs['maximum'][scenario_gdfs['maximum']['Sector'] == sector_name].copy()

    if min_sector.empty and max_sector.empty:
        print(f'Skipping {sector_name}: no features in either scenario')
        continue

    min_cap_usd = compute_scenario_cap_usd(min_sector, sector_name, 'minimum')
    max_cap_usd = compute_scenario_cap_usd(max_sector, sector_name, 'maximum')
    sector_shared_cap_usd = max(min_cap_usd, max_cap_usd)
    sector_shared_cap_usd = float(manual_sector_cap_usd.get(sector_name, sector_shared_cap_usd))

    unit_factor, unit_label = choose_units(sector_shared_cap_usd)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7.2), sharex=True, sharey=True)

    meta_min, out_min = plot_sector_panel(axes[0], min_sector, sector_name, 'minimum', sector_shared_cap_usd, unit_factor, unit_label)
    meta_max, out_max = plot_sector_panel(axes[1], max_sector, sector_name, 'maximum', sector_shared_cap_usd, unit_factor, unit_label)
    summary_rows.extend([meta_min, meta_max])

    fig.suptitle(
        f'{sector_name.capitalize()}: minimum vs maximum (shared sector scale = +/-{sector_shared_cap_usd / unit_factor:,.2f} {unit_label})',
        fontsize=13,
        y=0.97,
    )
    fig.subplots_adjust(top=0.955, bottom=0.08, wspace=0.06)

    out_png = out_dir / f'sector_{sector_name}_min_max_pair_panel_scales.png'
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    print('Saved map:', out_png)
    plt.show()

    # Outlier tables for auditing
    for scen, out_tbl in [('minimum', out_min), ('maximum', out_max)]:
        out_csv = out_dir / f'sector_{sector_name}_{scen}_outliers_q{int(display_quantile*1000)}_panel_scale.csv'
        if out_tbl is None or out_tbl.empty:
            pandas.DataFrame(columns=['Sector','Subsector','Asset','Layer','Asset_ID',value_col]).to_csv(out_csv, index=False)
        else:
            t = out_tbl.sort_values(value_col, ascending=False).copy()
            rp = t.geometry.representative_point()
            t['label_x'] = rp.x
            t['label_y'] = rp.y
            keep_cols = [
                'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', value_col,
                'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'label_x', 'label_y'
            ]
            t[keep_cols].to_csv(out_csv, index=False)
        print('Saved outliers:', out_csv)

summary_df = pandas.DataFrame(summary_rows)
summary_csv = out_dir / f'sector_panel_scale_summary_q{int(display_quantile*1000)}.csv'
summary_df.to_csv(summary_csv, index=False)
print('Saved summary:', summary_csv)
display(summary_df)
